
# Twin Prime Endpoint Discrepancy Engine v2

Dean Kulik / QuHarmonics — Nexus Primorial Clearance Algebra

This notebook instruments the current twin-prime proof hinge:

$$
D(n)=\min\{p>2:p\mid n(n+2)\}
$$

$$
\sigma(n)=\frac{D(n)}{\sqrt{n+2}}
$$

Twin primes are exactly:

$$
\boxed{\sigma(n)>1}
$$

Boundary contacts are fake witnesses:

$$
\boxed{\sigma(n)=1\ \not\Rightarrow\ \text{twin}}
$$

The engine works on the 15 twin-admissible rails modulo:

$$
W=210=2\cdot3\cdot5\cdot7
$$

where:

$$
\gcd(r,210)=1,\qquad \gcd(r+2,210)=1.
$$

## v2 upgrade

The v1 run showed the correct raw shape, but the square-root product term must be labeled correctly.

Correction:

$$
M_{\sqrt X}(X)=X\prod_{3\le p\le\sqrt{X+2}}\left(1-\frac{2}{p}\right)
$$

is a **naive endpoint-survival product diagnostic**, not the Hardy-Littlewood main term.

Use:

$$
M_{\mathrm{HL}}(X)\approx 2C_2\operatorname{Li}_2(X)
$$

for discrepancy diagnostics:

$$
E_2(X)=T_2(X)-M_{\mathrm{HL}}(X)
$$

$$
Z_2(X)=\frac{E_2(X)}{\sqrt{M_{\mathrm{HL}}(X)}}.
$$

New in this version:

1. cumulative discrepancy;
2. interval / annular discrepancy;
3. rail-internal $Z$;
4. rail-HL $Z$;
5. cancellation ratio $E/M$;
6. fake-witness boundary contacts;
7. rail-by-$X$ heatmap.


In [1]:

"""
Twin Prime Endpoint Discrepancy Engine v2
Dean Kulik / QuHarmonics — Nexus Primorial Clearance Algebra

This engine instruments the exact twin-prime hinge:

    D(n) = min{p > 2 : p | n(n+2)}
    sigma(n) = D(n) / sqrt(n+2)
    twin prime iff sigma(n) > 1

New in v2:
- separates cumulative and interval/annular discrepancy
- labels the square-root product as a naive survivor product, not the HL main term
- adds rail-internal Z and rail-HL Z
- adds fake-witness boundary classification at sigma = 1
- adds cancellation ratio E/M
- adds rail-by-interval heatmap tables for drift/cancellation auditing
"""

from __future__ import annotations

import math
import time
import json
from pathlib import Path
from typing import Dict, List, Iterable, Tuple, Optional

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

try:
    import scipy.stats as scipy_stats
except Exception:
    scipy_stats = None


CONFIG = {
    "W": 210,
    "X_GRID": [100_000, 1_000_000, 5_000_000, 20_000_000],
    "INTERVAL_BREAKS": [0, 100_000, 1_000_000, 5_000_000, 20_000_000],
    # For deeper local runs:
    # "X_GRID": [1_000_000, 5_000_000, 10_000_000, 20_000_000, 50_000_000, 100_000_000],
    # "INTERVAL_BREAKS": [0, 1_000_000, 5_000_000, 10_000_000, 20_000_000, 50_000_000, 100_000_000],
    "TAIL_THRESHOLDS": [0.35, 0.50, 0.75, 0.90, 0.95, 0.99, 1.00],
    "TAIL_CURVE_POINTS": np.round(np.linspace(0.05, 1.00, 40), 4).tolist(),
    "OUTPUT_DIR": "twin_prime_endpoint_discrepancy_v2_outputs",
    "TWIN_PRIME_C2": 0.6601618158468696,
    "LI2_SERIES_TERMS": 6,
}


# -----------------------------
# SIEVES
# -----------------------------

def prime_sieve_bool(n: int) -> np.ndarray:
    """Return boolean array is_prime[0..n]."""
    if n < 2:
        return np.zeros(n + 1, dtype=bool)
    sieve = np.ones(n + 1, dtype=bool)
    sieve[:2] = False
    sieve[4::2] = False
    root = int(math.isqrt(n))
    for p in range(3, root + 1, 2):
        if sieve[p]:
            sieve[p*p::2*p] = False
    return sieve


def spf_sieve(n: int) -> np.ndarray:
    """Smallest prime factor array spf[0..n]. For primes p, spf[p] = p."""
    spf = np.zeros(n + 1, dtype=np.int32)
    if n >= 1:
        spf[1] = 1
    if n >= 2:
        spf[2::2] = 2

    root = int(math.isqrt(n))
    for p in range(3, root + 1, 2):
        if spf[p] == 0:
            spf[p] = p
            start = p * p
            step = 2 * p
            view = spf[start:n+1:step]
            mask = (view == 0)
            view[mask] = p
            spf[start:n+1:step] = view

    zeros = (spf == 0)
    idx = np.nonzero(zeros)[0]
    spf[idx] = idx.astype(np.int32)
    spf[0] = 0
    return spf


# -----------------------------
# RAIL STRUCTURE
# -----------------------------

def twin_rails(W: int = 210) -> List[int]:
    """Twin-admissible rails modulo W: gcd(r,W)=gcd(r+2,W)=1."""
    return [
        r for r in range(1, W)
        if math.gcd(r, W) == 1 and math.gcd((r + 2) % W, W) == 1
    ]


def candidate_ns_for_rail(r: int, X: int, W: int = 210, lower_exclusive: int = 7) -> np.ndarray:
    """All lower_exclusive < n <= X with n == r mod W."""
    start_min = lower_exclusive + 1
    if r < start_min:
        start = r + W * ((start_min - r + W - 1) // W)
    else:
        start = r
    if start > X:
        return np.array([], dtype=np.int64)
    return np.arange(start, X + 1, W, dtype=np.int64)


def candidate_ns_for_rail_interval(r: int, A: int, B: int, W: int = 210, lower_exclusive: int = 7) -> np.ndarray:
    """All max(A, lower_exclusive) < n <= B with n == r mod W."""
    lo = max(A + 1, lower_exclusive + 1)
    if r < lo:
        start = r + W * ((lo - r + W - 1) // W)
    else:
        start = r
    if start > B:
        return np.array([], dtype=np.int64)
    return np.arange(start, B + 1, W, dtype=np.int64)


# -----------------------------
# MAIN TERMS
# -----------------------------

def li2_asymptotic_series(X: int | float, terms: int = 6) -> float:
    """
    Li_2(X) = integral_2^X dt/log(t)^2
    asymptotic ~ X/log(X)^2 * sum_{k>=0} (k+1)!/log(X)^k.
    """
    if X <= 2:
        return 0.0
    L = math.log(X)
    series = sum(math.factorial(k + 1) / (L ** k) for k in range(terms))
    return X / (L * L) * series


def twin_main_hl_series(X: int | float, C2: float = 0.6601618158468696, terms: int = 6) -> float:
    """Hardy-Littlewood cumulative estimate: M_HL(X) ~ 2 C2 Li_2(X)."""
    return 2.0 * C2 * li2_asymptotic_series(X, terms=terms)


def twin_main_hl_interval(A: int | float, B: int | float, C2: float = 0.6601618158468696, terms: int = 6) -> float:
    """Hardy-Littlewood interval estimate M(B)-M(A)."""
    return twin_main_hl_series(B, C2=C2, terms=terms) - twin_main_hl_series(A, C2=C2, terms=terms)


def twin_main_naive_sqrt_product(X: int) -> float:
    """
    Naive finite square-root-depth endpoint-survival product:
        X * prod_{3<=p<=sqrt(X+2)}(1-2/p)

    Correction: this is NOT the Hardy-Littlewood main term. It is intentionally kept
    as a diagnostic overestimate showing what independent endpoint clearance would
    predict before correlation/singular-series normalization.
    """
    y = int(math.isqrt(X + 2))
    primes_bool = prime_sieve_bool(y)
    primes = np.nonzero(primes_bool)[0]
    prod = 1.0
    for p in primes:
        if p >= 3:
            prod *= (1.0 - 2.0 / p)
    return X * prod


# -----------------------------
# CORE MEASUREMENT
# -----------------------------

def analyze_range(
    A: int,
    B: int,
    *,
    W: int = 210,
    is_prime: Optional[np.ndarray] = None,
    spf: Optional[np.ndarray] = None,
    tail_thresholds: Iterable[float] = (0.35, 0.50, 0.75, 0.90, 0.95, 0.99, 1.00),
    C2: float = 0.6601618158468696,
    li2_terms: int = 6,
) -> Dict:
    """
    Analyze n in (A, B] on the 15 twin rails.
    For cumulative use A=0.
    """
    t0 = time.time()
    if B < 10:
        raise ValueError("B must be >= 10 for twin rail analysis")
    if is_prime is None:
        is_prime = prime_sieve_bool(B + 2)
    if spf is None:
        spf = spf_sieve(B + 2)

    rails = twin_rails(W)

    rail_counts: Dict[int, int] = {}
    rail_candidates: Dict[int, int] = {}
    rail_tails: Dict[int, Dict[float, int]] = {}
    rail_boundary_contacts: Dict[int, int] = {}
    rail_near_miss_099_to_1: Dict[int, int] = {}

    total_candidates = 0
    total_tails = {float(t): 0 for t in tail_thresholds}
    boundary_contacts_total = 0

    for r in rails:
        ns = candidate_ns_for_rail_interval(r, A, B, W=W)
        total_candidates += len(ns)
        rail_candidates[r] = int(len(ns))

        if len(ns) == 0:
            rail_counts[r] = 0
            rail_tails[r] = {float(t): 0 for t in tail_thresholds}
            rail_boundary_contacts[r] = 0
            rail_near_miss_099_to_1[r] = 0
            continue

        twin_mask = is_prime[ns] & is_prime[ns + 2]
        count = int(np.sum(twin_mask))
        rail_counts[r] = count

        # first removal prime among endpoints
        D = np.minimum(spf[ns], spf[ns + 2]).astype(np.int64)
        sigma = D.astype(np.float64) / np.sqrt(ns + 2)

        this_tails = {}
        for t in tail_thresholds:
            c = int(np.sum(sigma > float(t)))
            this_tails[float(t)] = c
            total_tails[float(t)] += c
        rail_tails[r] = this_tails

        # fake witness at exactly sigma=1: normally n+2 = q^2 and n prime.
        bc_mask = (D * D == (ns + 2)) & (~twin_mask)
        bc = int(np.sum(bc_mask))
        rail_boundary_contacts[r] = bc
        boundary_contacts_total += bc

        rail_near_miss_099_to_1[r] = int(np.sum((sigma > 0.99) & (sigma <= 1.0)))

    T = int(sum(rail_counts.values()))
    values = np.array([rail_counts[r] for r in rails], dtype=float)
    internal_expected = np.full(len(rails), T / len(rails), dtype=float)

    chi2 = float(np.sum((values - internal_expected) ** 2 / internal_expected)) if T > 0 else float("nan")
    p_value = float(scipy_stats.chi2.sf(chi2, len(rails) - 1)) if scipy_stats is not None and T > 0 else None

    M_HL = twin_main_hl_interval(A, B, C2=C2, terms=li2_terms)
    E_HL = T - M_HL
    Z_HL = E_HL / math.sqrt(M_HL) if M_HL > 0 else float("nan")
    cancellation_ratio = E_HL / M_HL if M_HL > 0 else float("nan")

    per_rail_HL = M_HL / len(rails)
    per_rail_HL_sigma = math.sqrt(per_rail_HL) if per_rail_HL > 0 else float("nan")
    rail_Z_HL = {
        r: (rail_counts[r] - per_rail_HL) / per_rail_HL_sigma
        for r in rails
    }

    per_rail_internal = T / len(rails) if len(rails) else float("nan")
    per_rail_internal_sigma = math.sqrt(per_rail_internal) if per_rail_internal > 0 else float("nan")
    rail_Z_internal = {
        r: (rail_counts[r] - per_rail_internal) / per_rail_internal_sigma
        for r in rails
    }

    elapsed = time.time() - t0

    return {
        "A": int(A),
        "B": int(B),
        "W": int(W),
        "rails": rails,
        "num_rails": len(rails),
        "total_candidates": int(total_candidates),
        "T_twins_sigma_gt_1": T,
        "tail_counts": total_tails,
        "near_miss_099_to_1": int(total_tails.get(0.99, 0) - total_tails.get(1.00, 0)),
        "boundary_contacts_sigma_eq_1_not_twin": int(boundary_contacts_total),
        "rail_counts": rail_counts,
        "rail_candidates": rail_candidates,
        "rail_tails": rail_tails,
        "rail_boundary_contacts": rail_boundary_contacts,
        "rail_near_miss_099_to_1": rail_near_miss_099_to_1,
        "min_rail_count": int(values.min()) if len(values) else None,
        "max_rail_count": int(values.max()) if len(values) else None,
        "max_min_ratio": float(values.max() / values.min()) if len(values) and values.min() > 0 else None,
        "chi2_internal_uniform": chi2,
        "chi2_p_value_internal_uniform": p_value,
        "M_HL_interval": float(M_HL),
        "E_HL_interval": float(E_HL),
        "Z_HL_interval": float(Z_HL),
        "cancellation_ratio_E_over_M": float(cancellation_ratio),
        "rail_Z_HL": rail_Z_HL,
        "rail_Z_internal": rail_Z_internal,
        "elapsed_seconds": elapsed,
    }


def run_v2(config: Dict = CONFIG) -> Tuple[pd.DataFrame, pd.DataFrame, Dict]:
    """
    Runs cumulative X grid plus interval/annular breaks.
    Uses one sieve up to max endpoint for speed/consistency.
    """
    max_X = max(max(config["X_GRID"]), max(config["INTERVAL_BREAKS"]))
    print(f"Building sieves up to {max_X+2:,} ...")
    is_prime = prime_sieve_bool(max_X + 2)
    spf = spf_sieve(max_X + 2)

    raw = {"cumulative": {}, "intervals": {}}
    rows_cum = []
    print("Running cumulative grid...")
    for X in config["X_GRID"]:
        res = analyze_range(
            0, X,
            W=config["W"],
            is_prime=is_prime,
            spf=spf,
            tail_thresholds=config["TAIL_THRESHOLDS"],
            C2=config["TWIN_PRIME_C2"],
            li2_terms=config["LI2_SERIES_TERMS"],
        )
        raw["cumulative"][X] = res
        rows_cum.append(_row_from_result(res, label=f"<= {X:,}"))

    rows_int = []
    print("Running interval grid...")
    br = config["INTERVAL_BREAKS"]
    for A, B in zip(br[:-1], br[1:]):
        res = analyze_range(
            A, B,
            W=config["W"],
            is_prime=is_prime,
            spf=spf,
            tail_thresholds=config["TAIL_THRESHOLDS"],
            C2=config["TWIN_PRIME_C2"],
            li2_terms=config["LI2_SERIES_TERMS"],
        )
        raw["intervals"][(A, B)] = res
        rows_int.append(_row_from_result(res, label=f"({A:,}, {B:,}]"))

    return pd.DataFrame(rows_cum), pd.DataFrame(rows_int), raw


def _row_from_result(res: Dict, label: str) -> Dict:
    return {
        "range": label,
        "A": res["A"],
        "B": res["B"],
        "candidates": res["total_candidates"],
        "twins_sigma_gt_1": res["T_twins_sigma_gt_1"],
        "tail_gt_099": res["tail_counts"].get(0.99, None),
        "tail_gt_035": res["tail_counts"].get(0.35, None),
        "near_miss_099_to_1": res["near_miss_099_to_1"],
        "boundary_contacts": res["boundary_contacts_sigma_eq_1_not_twin"],
        "min_rail": res["min_rail_count"],
        "max_rail": res["max_rail_count"],
        "max_min_ratio": res["max_min_ratio"],
        "chi2": res["chi2_internal_uniform"],
        "chi2_p_value": res["chi2_p_value_internal_uniform"],
        "M_HL": res["M_HL_interval"],
        "E_HL": res["E_HL_interval"],
        "Z_HL": res["Z_HL_interval"],
        "E_over_M": res["cancellation_ratio_E_over_M"],
        "elapsed_seconds": res["elapsed_seconds"],
    }


# -----------------------------
# TABLES/PLOTS
# -----------------------------

def make_json_safe(obj):
    if isinstance(obj, dict):
        return {str(k): make_json_safe(v) for k, v in obj.items()}
    if isinstance(obj, list):
        return [make_json_safe(v) for v in obj]
    if isinstance(obj, tuple):
        return [make_json_safe(v) for v in obj]
    if isinstance(obj, (np.integer,)):
        return int(obj)
    if isinstance(obj, (np.floating,)):
        return float(obj)
    if isinstance(obj, np.ndarray):
        return obj.tolist()
    return obj


def rail_table_from_result(res: Dict, z_kind: str = "internal") -> pd.DataFrame:
    z_key = "rail_Z_internal" if z_kind == "internal" else "rail_Z_HL"
    return pd.DataFrame({
        "rail": res["rails"],
        "twin_count": [res["rail_counts"][r] for r in res["rails"]],
        "candidates": [res["rail_candidates"][r] for r in res["rails"]],
        "Z": [res[z_key][r] for r in res["rails"]],
        "near_miss_099_to_1": [res["rail_near_miss_099_to_1"][r] for r in res["rails"]],
        "boundary_contacts": [res["rail_boundary_contacts"][r] for r in res["rails"]],
    })


def save_outputs_v2(cum_df: pd.DataFrame, interval_df: pd.DataFrame, raw: Dict, output_dir: str) -> None:
    out = Path(output_dir)
    out.mkdir(parents=True, exist_ok=True)
    cum_df.to_csv(out / "cumulative_summary.csv", index=False)
    interval_df.to_csv(out / "interval_summary.csv", index=False)
    with open(out / "raw_results_v2.json", "w", encoding="utf-8") as f:
        json.dump(make_json_safe(raw), f, indent=2)

    # Last cumulative rail table
    last_X = max(raw["cumulative"].keys())
    rail_table_from_result(raw["cumulative"][last_X], "internal").to_csv(out / f"rail_internal_Z_X_{last_X}.csv", index=False)
    rail_table_from_result(raw["cumulative"][last_X], "HL").to_csv(out / f"rail_HL_Z_X_{last_X}.csv", index=False)


def plot_outputs_v2(cum_df: pd.DataFrame, interval_df: pd.DataFrame, raw: Dict, output_dir: str) -> None:
    out = Path(output_dir)
    out.mkdir(parents=True, exist_ok=True)

    plt.figure(figsize=(8, 5))
    plt.plot(cum_df["B"], cum_df["twins_sigma_gt_1"], marker="o", label="T observed")
    plt.plot(cum_df["B"], cum_df["M_HL"], marker="x", label="M_HL")
    plt.xscale("log"); plt.yscale("log")
    plt.xlabel("X")
    plt.ylabel("count")
    plt.title("Twin spillovers vs Hardy-Littlewood main term")
    plt.grid(True, which="both", alpha=0.3)
    plt.legend()
    plt.tight_layout()
    plt.savefig(out / "twins_vs_HL_main.png", dpi=160)
    plt.close()

    plt.figure(figsize=(8, 5))
    plt.plot(cum_df["B"], cum_df["E_over_M"], marker="o")
    plt.axhline(0, linewidth=1)
    plt.xscale("log")
    plt.xlabel("X")
    plt.ylabel("E_HL / M_HL")
    plt.title("Cumulative cancellation ratio")
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig(out / "cumulative_cancellation_ratio.png", dpi=160)
    plt.close()

    plt.figure(figsize=(8, 5))
    plt.plot(interval_df["B"], interval_df["Z_HL"], marker="o")
    plt.axhline(0, linewidth=1)
    plt.xscale("log")
    plt.xlabel("interval endpoint B")
    plt.ylabel("interval Z_HL")
    plt.title("Annular/interval endpoint discrepancy Z")
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig(out / "interval_Z_HL.png", dpi=160)
    plt.close()

    # Heatmap of internal rail Z by cumulative X
    rails = raw["cumulative"][max(raw["cumulative"].keys())]["rails"]
    matrix = []
    xlabels = []
    for X in sorted(raw["cumulative"].keys()):
        res = raw["cumulative"][X]
        matrix.append([res["rail_Z_internal"][r] for r in rails])
        xlabels.append(str(X))
    matrix = np.array(matrix).T

    plt.figure(figsize=(9, 5))
    plt.imshow(matrix, aspect="auto", interpolation="nearest")
    plt.colorbar(label="internal rail Z")
    plt.yticks(range(len(rails)), [str(r) for r in rails])
    plt.xticks(range(len(xlabels)), xlabels, rotation=45, ha="right")
    plt.xlabel("X")
    plt.ylabel("rail r mod 210")
    plt.title("Rail-local internal discrepancy heatmap")
    plt.tight_layout()
    plt.savefig(out / "rail_internal_Z_heatmap.png", dpi=160)
    plt.close()


def print_lock_readout(cum_df: pd.DataFrame, interval_df: pd.DataFrame) -> None:
    print("\nΔ LOCK READOUT")
    last = cum_df.iloc[-1]
    print(f"Last X = {int(last['B']):,}")
    print(f"Twin spillovers σ>1 = {int(last['twins_sigma_gt_1']):,}")
    print(f"HL main term M = {last['M_HL']:.6f}")
    print(f"Endpoint discrepancy E = {last['E_HL']:.6f}")
    print(f"Z = {last['Z_HL']:.6f}")
    print(f"E/M = {last['E_over_M']:.6e}")
    print(f"Rail max/min = {last['max_min_ratio']:.6f}, chi2 p = {last['chi2_p_value']:.6f}")
    print("\nCorrection: M_naive_sqrt_product from v1 is an overestimate diagnostic, not the proof main term.")
    print("Use M_HL = 2 C2 Li_2(X) for endpoint-discrepancy Z.")


if __name__ == "__main__":
    print("Twin Prime Endpoint Discrepancy Engine v2")
    print("Rails:", twin_rails(CONFIG["W"]))
    cum_df, interval_df, raw = run_v2(CONFIG)
    print("\nCumulative summary:")
    print(cum_df)
    print("\nInterval summary:")
    print(interval_df)
    save_outputs_v2(cum_df, interval_df, raw, CONFIG["OUTPUT_DIR"])
    plot_outputs_v2(cum_df, interval_df, raw, CONFIG["OUTPUT_DIR"])
    print_lock_readout(cum_df, interval_df)
    print(f"\nOutputs saved to {CONFIG['OUTPUT_DIR']}")


Twin Prime Endpoint Discrepancy Engine v2
Rails: [11, 17, 29, 41, 59, 71, 101, 107, 137, 149, 167, 179, 191, 197, 209]
Building sieves up to 20,000,002 ...
Running cumulative grid...
Running interval grid...

Cumulative summary:
           range  A         B  candidates  twins_sigma_gt_1  tail_gt_099  \
0     <= 100,000  0    100000        7143              1222         1253   
1   <= 1,000,000  0   1000000       71427              8167         8247   
2   <= 5,000,000  0   5000000      357143             32461        32689   
3  <= 20,000,000  0  20000000     1428572            107405       108032   

   tail_gt_035  near_miss_099_to_1  boundary_contacts  min_rail  max_rail  \
0         2342                  31                 23        73        95   
1        14339                  80                 48       518       572   
2        53817                 228                 87      2114      2220   
3       171554                 627                141      7062      7261   

   m


## Run v2 grid

This produces two tables:

$$
\text{cumulative: }(0,X]
$$

and

$$
\text{interval: }(A,B].
$$

The interval table is the important new object because permanent cancellation would need to persist locally, not merely cumulatively.


In [2]:

cum_df, interval_df, raw = run_v2(CONFIG)
cum_df


Building sieves up to 20,000,002 ...
Running cumulative grid...
Running interval grid...


,range,A,B,candidates,twins_sigma_gt_1,tail_gt_099,tail_gt_035,near_miss_099_to_1,boundary_contacts,min_rail,max_rail,max_min_ratio,chi2,chi2_p_value,M_HL,E_HL,Z_HL,E_over_M,elapsed_seconds
0,"<= 100,000",0,100000,7143,1222,1253,2342,31,23,73,95,1.301370,5.692308,0.973661,1240.263241,-18.263241,-0.518586,-0.014725,0.000998
1,"<= 1,000,000",0,1000000,71427,8167,8247,14339,80,48,518,572,1.104247,6.203747,0.961092,8231.954724,-64.954724,-0.715911,-0.007891,0.003000
2,"<= 5,000,000",0,5000000,357143,32461,32689,53817,228,87,2114,2220,1.050142,4.942054,0.986603,32306.563383,154.436617,0.859221,0.004780,0.011501
3,"<= 20,000,000",0,20000000,1428572,107405,108032,171554,627,141,7062,7261,1.028179,6.678926,0.946381,107197.147797,207.852203,0.634838,0.001939,0.060500


In [3]:

interval_df


,range,A,B,candidates,twins_sigma_gt_1,tail_gt_099,tail_gt_035,near_miss_099_to_1,boundary_contacts,min_rail,max_rail,max_min_ratio,chi2,chi2_p_value,M_HL,E_HL,Z_HL,E_over_M,elapsed_seconds
0,"(0, 100,000]",0,100000,7143,1222,1253,2342,31,23,73,95,1.301370,5.692308,0.973661,1240.263241,-18.263241,-0.518586,-0.014725,0.000999
1,"(100,000, 1,000,000]",100000,1000000,64284,6945,6994,11997,49,25,432,493,1.141204,6.725702,0.944769,6991.691483,-46.691483,-0.558401,-0.006678,0.003001
2,"(1,000,000, 5,000,000]",1000000,5000000,285716,24294,24442,39478,148,39,1573,1664,1.057851,4.419363,0.992368,24074.608659,219.391341,1.413969,0.009113,0.009498
3,"(5,000,000, 20,000,000]",5000000,20000000,1071429,74944,75343,117737,399,54,4892,5083,1.039043,8.631431,0.853912,74890.584414,53.415586,0.195189,0.000713,0.048001



## Save outputs and plots


In [4]:

save_outputs_v2(cum_df, interval_df, raw, CONFIG["OUTPUT_DIR"])
plot_outputs_v2(cum_df, interval_df, raw, CONFIG["OUTPUT_DIR"])
print_lock_readout(cum_df, interval_df)



Δ LOCK READOUT
Last X = 20,000,000
Twin spillovers σ>1 = 107,405
HL main term M = 107197.147797
Endpoint discrepancy E = 207.852203
Z = 0.634838
E/M = 1.938971e-03
Rail max/min = 1.028179, chi2 p = 0.946381

Correction: M_naive_sqrt_product from v1 is an overestimate diagnostic, not the proof main term.
Use M_HL = 2 C2 Li_2(X) for endpoint-discrepancy Z.



## Inspect the last rail table

Internal $Z$ checks rail-to-rail equidistribution after conditioning on the observed total:

$$
Z_{r,\mathrm{int}}=
\frac{T_r-T/15}{\sqrt{T/15}}.
$$

HL $Z$ checks absolute discrepancy against the Hardy-Littlewood main term:

$$
Z_{r,\mathrm{HL}}=
\frac{T_r-M_{\mathrm{HL}}/15}{\sqrt{M_{\mathrm{HL}}/15}}.
$$


In [5]:

last_X = max(raw["cumulative"].keys())
rail_internal = rail_table_from_result(raw["cumulative"][last_X], "internal")
rail_HL = rail_table_from_result(raw["cumulative"][last_X], "HL")
rail_internal, rail_HL


(    rail  twin_count  candidates         Z  near_miss_099_to_1  \
 0     11        7261       95239  1.189650                  36   
 1     17        7181       95239  0.244233                  34   
 2     29        7090       95238 -0.831179                  30   
 3     41        7221       95238  0.716941                  39   
 4     59        7149       95238 -0.133934                  39   
 5     71        7185       95238  0.291504                  49   
 6    101        7116       95238 -0.523919                  36   
 7    107        7194       95238  0.397863                  63   
 8    137        7163       95238  0.031514                  28   
 9    149        7062       95238 -1.162075                  54   
 10   167        7169       95238  0.102420                  56   
 11   179        7167       95238  0.078785                  22   
 12   191        7241       95238  0.953295                  26   
 13   197        7134       95238 -0.311200                  4